<a href="https://colab.research.google.com/github/supsi-dacd-isaac/TeachDecisionMakingUncertainty/blob/main/excercice_model_fitting_and_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Uncertainty - modelling and slection**

This Colab notebook provides an introduction to uncertainty modeling, covering aleatoric models and an introduction to model selection.  

**THE STUDENTS WILL**:
1. Generate n=1000 data points using the DGM provided
2. Explore the data (plot it, print mean, standard deviation)
3. Fit a (parametric) GMM model and a (non-parametric) KDE model
4. Repeat the GMM fitting for a list of [2, 10, 15, 20, 60, 80] components. compare likelihood, BIC, and AIC.
5. Same for the KDE different bandwidth. 6. Visualization + discussion on the best model


## 🎯 **Data Generation Mechanism (DGM)**  

Before working with probabilistic models, we need some data.

We will start with two synthetic datasets.  

### 🔹 `data_generation_mechanism()` function  
This function simulates an experiment where samples are drawn and datasets generated.

We have three different sampling mechanisms:  

1️⃣ **Normal Distribution** – Generates Gaussian-distributed data.  
2️⃣ **Blobs (Clusters of Points)** – Useful for classification tasks.  
3️⃣ **Two Moons Dataset** – A non-linearly separable dataset often used for clustering.   

---

In [6]:
# Here we import useful packages and methods
# math and statistical tools
import numpy as np
import math
import scipy.stats as stats
import numpy.random as random

# tools for visualization and plotting
import seaborn as sns
import matplotlib.pyplot as plt

import sklearn as skl
from sklearn.datasets import make_blobs, make_moons
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity

In [7]:
def data_generation_mechanism(n_samples: int = 100,
                              n_dimensions: int = 1,
                              type='normal',
                              add_noise: bool = False,
                              nose_level: float = 0.5):
    """
    Generates a matrix of random data with specified number of samples and dimensions.

    Parameters:
    - n_samples (int): Number of samples (rows of the matrix).
    - n_dimensions (int): Number of dimensions (columns of the matrix).
    - type (str): The type of data distribution ('normal', 'blobs', 'two_moons').

    Returns:
    - np.ndarray: A matrix with shape (n_samples, n_dimensions), filled with random data.
    """
    # Default to 'normal' if no type is specified
    if type is None:
        type = 'normal'

    if type == 'normal':
        # For normal distribution, we use a multivariate normal distribution
        mean = [0] * n_dimensions  # Mean vector
        cov = np.eye(n_dimensions)  # Identity covariance matrix (no correlation)
        data_matrix = np.random.multivariate_normal(mean, cov, size=n_samples)

    elif type == 'blobs':
        # Generate data from blobs (clusters of points)
        data_matrix, _ = make_blobs(n_samples=n_samples, n_features=n_dimensions, random_state=42)

    elif type == 'two_moons':
        # Generate two moons dataset (non-linear)
        data_matrix, _ = make_moons(n_samples=n_samples, noise=0.1, random_state=42)

    else:
        raise ValueError(f"Unsupported data type: {type}")


    if add_noise:
        # Add Gaussian noise to the data
        noise = np.random.normal(0, nose_level, size=(n_samples, n_dimensions))
        data_matrix += noise

    return data_matrix



## 🔬 **Data Visualization Examples**  

We generate and visualize different datasets using:  
- **Clusters of points (blobs)**  
- **Two moons dataset** (with and without noise)  

Additionally, we explore the effect of different **noise levels** on data distribution.  



In [8]:
# Data Generation Mechanism:
n_samples = 1000  # Number of samples
n_dimensions = 2  # Number of dimensions
data_blobs = data_generation_mechanism(n_samples,
                                       n_dimensions,
                                       type='blobs')      # Generate data of type 'blobs'
data_moons = data_generation_mechanism(n_samples,
                                       n_dimensions,
                                       type='two_moons',
                                       add_noise=False)


### 📊 **Plots Included**  
✔️ Scatter plots of generated data  
✔️ Effect of noise on the two moons dataset     

In [9]:
# Scatter plot visualization

# Generating mechanism + additional noise
noise_levels = [0.01, 0.05, 0.1, 0.5]



## **Fitting, visualization & comparison of models**:

**Raw Data:** Displays the actual points generated from the "Two Moons" dataset, which may have non-linear separability.

**GMM PDF:** Fit and visualized how well the Gaussian Mixture Model approximates the underlying data distribution with $M$ Gaussian components.

```
n_components = 2 # two gaussian mixtures
GMM = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
GMM.fit(data) # fit the model
```

**KDE Density:** Displays the smooth non-parametric estimate of the data density, showing how KDE adapts to the underlying data without assuming a parametric model.

```
  kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth)
  kde.fit(data)
```






**Iso-Probability Curves:** Helps compare how GMM and KDE models capture data distributions and their density levels.


```
  # Create a mesh grid for density visualization
  n_grid_points=200
  x = np.linspace(min(data[:, 0])-1, max(data[:, 0])+1, n_grid_points)
  y = np.linspace(min(data[:, 1])-1, max(data[:, 1])+1, n_grid_points)
  X, Y = np.meshgrid(x, y)
  grid_points = np.column_stack([X.ravel(), Y.ravel()])

  # Evaluate densities
  log_likelihood = model.score_samples(grid_points)
  pdf_density = np.exp(log_likelihood).reshape(X.shape)

  contour(X, Y, pdf_density, levels=20, colors='blue', linestyles='solid')

```





**Model selection:** Compare AIC, BIC, and likelihood of different models. Select a list of bandwidths (KDE) and list with a number of mixtures (GMM), repeat the fitting, evaluate goodness of fit vs complexity.


```
  kde.aic(data) # on fitted model
  kde.bic(data)
```